[![pythonista.io](img/pythonista.png)](https://www.pythonista.io)

# Despliegue seguro en *Linux*.



Este cuaderno aterriza la operación de despliegue en entorno *Linux* para fines de test y demostración, manteniendo a *GitHub Actions* como plano de control y reservando producción para *GCP*.



## Objetivos.



- Operar despliegues remotos por SSH con controles mínimos de seguridad.

- Aplicar la estrategia de *runners* del curso: *hosted* por defecto y *self-hosted* solo por necesidad real.

- Conectar trazabilidad de *pipeline* con cambios sobre hosts *Linux*.


## Controles mínimos para entorno *Linux* de test.



Para un despliegue serio en *Linux* se requiere al menos:



- autenticación remota acotada y rotación de credenciales;

- separación clara entre entornos de test y prod;

- secretos fuera del repositorio;

- pasos de despliegue idempotentes;

- evidencia posterior al cambio (*logs*, versión desplegada, validación básica).



En este curso, *Linux* en test no compite con prod en *GCP*: complementa la formación operativa en hosts reales.


## Ejemplo conceptual: despliegue remoto en *Linux* de test.



```yaml

deploy-test-linux:

  environment: test

  runs-on: [self-hosted, linux, test-net]

  steps:

    - name: Checkout

      uses: actions/checkout@v6



    - name: Configure SSH

      run: |

        install -m 700 -d ~/.ssh

        echo "$SSH_PRIVATE_KEY" > ~/.ssh/id_ed25519

        chmod 600 ~/.ssh/id_ed25519

        ssh-keyscan -H "$DEPLOY_HOST" >> ~/.ssh/known_hosts



    - name: Deploy to Linux test server

      run: |

        scp docker-compose.yml "$DEPLOY_USER@$DEPLOY_HOST:/srv/app/"

        ssh "$DEPLOY_USER@$DEPLOY_HOST" "cd /srv/app && docker compose pull && docker compose up -d"

```



El objetivo didáctico es mostrar que el despliegue por SSH también debe estar gobernado por entorno, identidad y trazabilidad, igual que cualquier otro cambio de *pipeline*.


## Flujo operativo recomendado.



Cuando el destino es un servidor *Linux* de test, el *pipeline* necesita una ruta de despliegue remoto por SSH. Esa ruta no debe tratarse como un simple `ssh user@host comando`, sino como una operación controlada con identidad, verificación de host, secretos acotados y pasos idempotentes.



Un flujo sano suele incluir:



- preparación del host con usuarios y permisos mínimos;

- distribución segura de credenciales o certificados temporales;

- verificación de `known_hosts`;

- actualización del artefacto;

- reinicio controlado del servicio;

- validación posterior al despliegue.



En paralelo, prod puede mantenerse en *GCP* con autenticación federada y sin llaves estáticas, evitando mezclar responsabilidades de operación entre entornos.


In [ ]:
flujo_entornos = [

    ('test-linux', 'self-hosted-justificado'),

    ('prod-gcp', 'hosted-oidc-o-runner-controlado')

]



for entorno, estrategia in flujo_entornos:

    print(f'entorno: {entorno} | estrategia: {estrategia}')


## Cierre del bloque operativo.



Este cuaderno completa la secuencia operativa del curso:



- construcción de artefactos de forma reproducible;

- gobierno de cambios de infraestructura con `plan` y `apply`;

- despliegue controlado en *Linux* de test sin romper la estrategia de prod en *GCP*.


<p style="text-align: center"><a rel="license" href="http://creativecommons.org/licenses/by/4.0/"><img alt="Licencia Creative Commons" style="border-width:0" src="https://i.creativecommons.org/l/by/4.0/80x15.png" /></a><br />Esta obra está bajo una <a rel="license" href="http://creativecommons.org/licenses/by/4.0/">Licencia Creative Commons Atribución 4.0 Internacional</a>.</p>
<p style="text-align: center">&copy; José Luis Chiquete Valdivieso. 2017-2026.</p>